# Cross-Source Comparison

Compare propagation data across WSPR, RBN, Contest, and PSK Reporter sources.

**What you'll learn:**
- Understand the differences between data sources
- Compare SNR distributions and coverage
- Identify source-specific biases

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ionis_jupyter import load_dataset, list_datasets

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
# Check which datasets are available
available = list_datasets()
print("Available datasets:")
for name, info in available.items():
    if info["exists"] and info["rows"]:
        print(f"  {name}: {info['rows']:,} rows")

## Data Source Characteristics

| Source | Mode | SNR Type | Bias |
|--------|------|----------|------|
| **WSPR** | WSPR | Machine-decoded (-30 to +20 dB) | Continuous, calibrated, global |
| **RBN** | CW/RTTY | Machine-decoded (8-29 dB) | Only decoded signals, skimmer coverage |
| **Contest** | SSB/RTTY | Anchored (+10/0 dB) | Contest hours only, success bias |
| **PSKR** | FT8/FT4/CW | Machine-decoded (-34 to +38 dB) | Dominant FT8, high volume |

In [ ]:
# Load available sources
sources = {}

for name in ["wspr", "rbn", "contest", "pskr"]:
    try:
        df = load_dataset(name, limit=100000)  # Limit for demo
        sources[name] = df
        print(f"Loaded {name}: {len(df):,} rows")
    except FileNotFoundError:
        print(f"{name}: not downloaded")

print(f"\nComparing {len(sources)} sources")

In [ ]:
# SNR distribution comparison
fig, ax = plt.subplots(figsize=(12, 6))

colors = {"wspr": "blue", "rbn": "green", "contest": "orange", "pskr": "red"}

for name, df in sources.items():
    snr = df["median_snr"].dropna()
    ax.hist(snr, bins=50, alpha=0.5, label=f"{name.upper()} (n={len(snr):,})",
            color=colors.get(name, "gray"), density=True)

ax.axvline(0, color="gray", linestyle="--", alpha=0.7)
ax.set_xlabel("Median SNR (dB)")
ax.set_ylabel("Density")
ax.set_title("SNR Distribution by Source")
ax.legend()
ax.set_xlim(-35, 25)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
stats = []
for name, df in sources.items():
    snr = df["median_snr"].dropna()
    stats.append({
        "Source": name.upper(),
        "Count": len(snr),
        "Mean SNR": snr.mean(),
        "Std SNR": snr.std(),
        "Min SNR": snr.min(),
        "Max SNR": snr.max(),
        "% Negative": (snr < 0).mean() * 100,
    })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

In [ ]:
# Band distribution by source
BAND_NAMES = {
    102: "160m", 103: "80m", 105: "40m", 107: "20m", 
    109: "15m", 111: "10m"
}

fig, axes = plt.subplots(1, len(sources), figsize=(4*len(sources), 5))
if len(sources) == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, sources.items()):
    band_counts = df.groupby("band").size().sort_index()
    bands = [BAND_NAMES.get(b, str(b)) for b in band_counts.index]
    ax.barh(bands, band_counts.values, color=colors.get(name, "gray"), alpha=0.7)
    ax.set_xlabel("Count")
    ax.set_title(f"{name.upper()}")

plt.suptitle("Band Distribution by Source", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Hour distribution by source
fig, ax = plt.subplots(figsize=(14, 6))

width = 0.2
x = np.arange(24)

for i, (name, df) in enumerate(sources.items()):
    hour_counts = df.groupby("hour").size()
    hour_counts = hour_counts.reindex(range(24), fill_value=0)
    hour_pct = hour_counts / hour_counts.sum() * 100
    ax.bar(x + i*width, hour_pct.values, width, 
           label=name.upper(), color=colors.get(name, "gray"), alpha=0.7)

ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("% of Signatures")
ax.set_title("Hour Distribution by Source")
ax.set_xticks(x + width * (len(sources)-1) / 2)
ax.set_xticklabels(range(24))
ax.legend()
plt.tight_layout()
plt.show()

## Key Differences

**WSPR:**
- Calibrated SNR from machine decoding
- Continuous 24/7 operation
- Lower power (typically 5W or less)
- Best for weak-signal propagation research

**RBN:**
- CW/RTTY only (no voice modes)
- Higher SNR floor (skimmer sensitivity limit)
- Contest-correlated activity spikes
- Good geographic coverage via skimmer network

**Contest:**
- Anchored SNR (+10 dB success, 0 dB marginal)
- Only during contest weekends
- Success bias (no "failed QSO" observations)
- SSB and RTTY modes

**PSK Reporter:**
- Dominated by FT8 (~88%)
- Very high volume (26M spots/day)
- Similar SNR quality to WSPR for digital modes
- Forward-only (no historical bulk download)